# Extracción de Datos

In [7]:
from pathlib import Path
import pandas as pd

RUTA_CSV = Path("C:\\Users\\ACER NITRO V16\\Documents\\Ciencia de Datos\\Proyecto1\\Ciencias-de-datos\\siniestralidad_2016_2024.csv")

if not RUTA_CSV.exists():
    raise FileNotFoundError(f"No se encontró el archivo fuente: {RUTA_CSV.resolve()}")

df_raw = pd.read_csv(
    RUTA_CSV,
    sep=";",
    encoding="latin-1",
    dtype="string",
    na_values=[".", ""],
    low_memory=False
)

print(f"Registros cargados: {len(df_raw):,}")
print("Columnas:")
print(df_raw.columns.tolist())

display(df_raw.head())
display(df_raw.isna().sum())

Registros cargados: 371,003
Columnas:
['Código', 'Fecha', 'Tipo confirmado', 'Dirección reporte', 'Tipo clase de accidente', 'Tipo clase de accidente.1', 'Tipo confirmado.1', 'Medio de reporte', 'Numero vehículos', 'Tipo de vehículos implicados']


,Código,Fecha,Tipo confirmado,Dirección reporte,Tipo clase de accidente,Tipo clase de accidente.1,Tipo confirmado.1,Medio de reporte,Numero vehículos,Tipo de vehículos implicados
0,96159,1/01/2016,Solo daños,Carrera 56 Con Calle 1 A,<NA>,<NA>,Solo Da?os,RADIO,2,MOTOCICLETA
1,96160,1/01/2016,Negativo,Calle 72 O Con Carrera 28 H,<NA>,<NA>,Negativo,RADIO,<NA>,<NA>
2,96161,1/01/2016,Con lesionado,Carrera 94 Con Calle 4 C,ATROPELLO,ATROPELLO,Por Atropello,TELEF?NICO 127,2,MOTOCICLETA
3,96162,1/01/2016,Con lesionado,Carrera 8 A Con Calle 44,VOLCAMIENTO,VOLCAMIENTO,Por Volcamiento,RADIO,1,MOTOCICLETA
4,96163,1/01/2016,Negativo,Carrera 28 H Con Calle 72 M,<NA>,<NA>,Negativo,TELEF?NICO 127,<NA>,<NA>


Código                               1
Fecha                                1
Tipo confirmado                      3
Dirección reporte               183686
Tipo clase de accidente         325258
Tipo clase de accidente.1       325258
Tipo confirmado.1                23389
Medio de reporte                288099
Numero vehículos                194424
Tipo de vehículos implicados    222214
dtype: int64

# Transformación del DataFrame

In [8]:
df = df_raw.copy()

df = df.rename(columns={
    "Código": "id_siniestro",
    "Fecha": "fecha",
    "Tipo confirmado": "tipo_confirmado_1",
    "Dirección reporte": "direccion_reporte",
    "Tipo clase de accidente": "tipo_accidente_1",
    "Tipo clase de accidente.1": "tipo_accidente_2",
    "Tipo confirmado.1": "tipo_confirmado_2",
    "Medio de reporte": "medio_reporte",
    "Numero vehículos": "numero_vehiculos",
    "Tipo de vehículos implicados": "tipos_vehiculos"
})

# Unifica las columnas duplicadas usando el primer valor disponible
df["tipo_accidente_raw"] = df["tipo_accidente_1"].combine_first(
    df["tipo_accidente_2"]
)

df["tipo_confirmado_raw"] = df["tipo_confirmado_1"].combine_first(
    df["tipo_confirmado_2"]
)

# Elimina las columnas duplicadas originales
df = df.drop(columns=[
    "tipo_accidente_1",
    "tipo_accidente_2",
    "tipo_confirmado_1",
    "tipo_confirmado_2"
])

# Conversión de tipos
df["id_siniestro"] = df["id_siniestro"].astype("string")
df["fecha"] = pd.to_datetime(
    df["fecha"],
    format="%d/%m/%Y",
    errors="coerce"
)
df["numero_vehiculos"] = pd.to_numeric(
    df["numero_vehiculos"],
    errors="coerce"
).astype("Int64")

print(df.dtypes)
display(df.head())

id_siniestro                   string
fecha                  datetime64[us]
direccion_reporte              string
medio_reporte                  string
numero_vehiculos                Int64
tipos_vehiculos                string
tipo_accidente_raw             string
tipo_confirmado_raw            string
dtype: object


,id_siniestro,fecha,direccion_reporte,medio_reporte,numero_vehiculos,tipos_vehiculos,tipo_accidente_raw,tipo_confirmado_raw
0,96159,2016-01-01,Carrera 56 Con Calle 1 A,RADIO,2,MOTOCICLETA,<NA>,Solo daños
1,96160,2016-01-01,Calle 72 O Con Carrera 28 H,RADIO,<NA>,<NA>,<NA>,Negativo
2,96161,2016-01-01,Carrera 94 Con Calle 4 C,TELEF?NICO 127,2,MOTOCICLETA,ATROPELLO,Con lesionado
3,96162,2016-01-01,Carrera 8 A Con Calle 44,RADIO,1,MOTOCICLETA,VOLCAMIENTO,Con lesionado
4,96163,2016-01-01,Carrera 28 H Con Calle 72 M,TELEF?NICO 127,<NA>,<NA>,<NA>,Negativo


# Control de Duplicados y Claves

In [9]:
# Eliminar filas completamente idénticas
df = df.drop_duplicates().copy()

# Eliminar registros sin identificador de siniestro
df = df.dropna(subset=["id_siniestro"]).copy()

# Conservar un solo registro por siniestro, priorizando el más completo
df["_completitud"] = df.notna().sum(axis=1)

df = (
    df.sort_values("_completitud", ascending=False)
    .drop_duplicates("id_siniestro", keep="first")
    .drop(columns="_completitud")
    .copy()
)

print(f"Registros después de la limpieza: {len(df):,}")
print(
    "Códigos duplicados:",
    df["id_siniestro"].duplicated().sum()
)
print(
    "Códigos nulos:",
    df["id_siniestro"].isna().sum()
)

Registros después de la limpieza: 370,998
Códigos duplicados: 0
Códigos nulos: 0


# Normalización de Textos

In [10]:
columnas_texto = [
    "direccion_reporte",
    "tipo_accidente_raw",
    "tipo_confirmado_raw",
    "medio_reporte",
    "tipos_vehiculos"
]

for columna in columnas_texto:
    df[columna] = (
        df[columna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )

display(df.head())

,id_siniestro,fecha,direccion_reporte,medio_reporte,numero_vehiculos,tipos_vehiculos,tipo_accidente_raw,tipo_confirmado_raw
370987,AI013532,2024-11-02,CALLE 9C & CARRERA 29A,AGENTE CAD 123,2,"MOTOCICLETA,AUTOMOVIL",CHOQUE,NEGATIVO
2,96161,2016-01-01,CARRERA 94 CON CALLE 4 C,TELEF?NICO 127,2,MOTOCICLETA,ATROPELLO,CON LESIONADO
370988,AI013533,2024-11-02,CALLE 16 & CARRERA 50,CAT 127,2,"AUTOMOVIL,MOTOCICLETA",CHOQUE,CON LESIONADO
370980,AI013525,2024-11-02,CALLE 73 & CARRERA 26F,CAT 127,2,"MOTOCICLETA,MOTOCICLETA",CHOQUE,NEGATIVO
370985,AI013530,2024-11-02,CALLE 46 & CARRERA 43B,CAT 127,2,"BUS,MOTOCICLETA",CHOQUE,CON LESIONADO


# Estandarizar las Categorías

In [11]:
import numpy as np

texto_gravedad = df["tipo_confirmado_raw"].fillna("")
texto_accidente = df["tipo_accidente_raw"].fillna("")
texto_medio = df["medio_reporte"].fillna("")

# Estandarización de la gravedad
df["gravedad_estandarizada"] = np.select(
    [
        texto_gravedad.str.contains("FALLEC", na=False),
        texto_gravedad.str.contains("LESION|HERID", na=False),
        texto_gravedad.str.contains("SOLO DA", na=False),
        texto_gravedad.str.contains("NEGATIV", na=False),
        df["tipo_confirmado_raw"].isna()
    ],
    [
        "CON FALLECIDOS",
        "CON HERIDOS",
        "SOLO DANOS",
        "NEGATIVO",
        "SIN DATO"
    ],
    default="OTRO"
)

# Estandarización del tipo de accidente
df["tipo_accidente_estandarizado"] = np.select(
    [
        texto_accidente.str.contains("ATROPEL", na=False),
        texto_accidente.str.contains("VOLCAM", na=False),
        texto_accidente.str.contains("CHOQUE", na=False),
        texto_accidente.str.contains("CA.?DA.*OCUPANTE", na=False),
        texto_accidente.str.contains("INCEND", na=False),
        df["tipo_accidente_raw"].isna()
    ],
    [
        "ATROPELLO",
        "VOLCAMIENTO",
        "CHOQUE",
        "CAIDA DE OCUPANTE",
        "INCENDIO",
        "SIN DATO"
    ],
    default="OTRO"
)

# Estandarización del medio de reporte
df["medio_reporte_estandarizado"] = np.select(
    [
        texto_medio.str.contains("123", na=False),
        texto_medio.str.contains("127", na=False),
        texto_medio.str.contains("RADIO", na=False),
        texto_medio.str.contains("AGENTE", na=False),
        texto_medio.str.contains("RED", na=False),
        df["medio_reporte"].isna()
    ],
    [
        "LINEA 123",
        "LINEA 127",
        "RADIO",
        "AGENTE DE TRANSITO",
        "REDES SOCIALES",
        "SIN DATO"
    ],
    default="OTRO-NARRATIVA"
)

display(df[
    [
        "tipo_confirmado_raw",
        "gravedad_estandarizada",
        "tipo_accidente_raw",
        "tipo_accidente_estandarizado",
        "medio_reporte",
        "medio_reporte_estandarizado"
    ]
].head(10))

,tipo_confirmado_raw,gravedad_estandarizada,tipo_accidente_raw,tipo_accidente_estandarizado,medio_reporte,medio_reporte_estandarizado
370987,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,AGENTE CAD 123,LINEA 123
2,CON LESIONADO,CON HERIDOS,ATROPELLO,ATROPELLO,TELEF?NICO 127,LINEA 127
370988,CON LESIONADO,CON HERIDOS,CHOQUE,CHOQUE,CAT 127,LINEA 127
370980,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,CAT 127,LINEA 127
370985,CON LESIONADO,CON HERIDOS,CHOQUE,CHOQUE,CAT 127,LINEA 127
3,CON LESIONADO,CON HERIDOS,VOLCAMIENTO,VOLCAMIENTO,RADIO,RADIO
371001,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,AGENTE RADIO,RADIO
314342,NEGATIVO,NEGATIVO,CHOQUE,CHOQUE,TELEF?NICO 127,LINEA 127
314344,SOLO DAÑOS,SOLO DANOS,CHOQUE,CHOQUE,TELEF?NICO 127,LINEA 127
46,CON LESIONADO,CON HERIDOS,ATROPELLO,ATROPELLO,CAD 123,LINEA 123


# Dimensión de Tiempo

In [13]:
dim_tiempo = (
    df[["fecha"]]
    .dropna()
    .drop_duplicates()
    .sort_values("fecha")
    .reset_index(drop=True)
)

dim_tiempo["id_tiempo"] = (
    dim_tiempo["fecha"].dt.strftime("%Y%m%d").astype(int)
)

dim_tiempo["anio"] = dim_tiempo["fecha"].dt.year
dim_tiempo["trimestre"] = dim_tiempo["fecha"].dt.quarter
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month
dim_tiempo["nombre_mes"] = dim_tiempo["fecha"].dt.month_name()
dim_tiempo["dia"] = dim_tiempo["fecha"].dt.day
dim_tiempo["dia_semana_num"] = dim_tiempo["fecha"].dt.dayofweek + 1
dim_tiempo["nombre_dia_semana"] = dim_tiempo["fecha"].dt.day_name()
dim_tiempo["es_fin_de_semana"] = (
    dim_tiempo["fecha"].dt.dayofweek >= 5
)
dim_tiempo["semana_anio"] = (
    dim_tiempo["fecha"].dt.isocalendar().week.astype(int)
)

dim_tiempo = dim_tiempo[
    [
        "id_tiempo",
        "fecha",
        "anio",
        "trimestre",
        "mes",
        "nombre_mes",
        "dia",
        "dia_semana_num",
        "nombre_dia_semana",
        "es_fin_de_semana",
        "semana_anio"
    ]
]

display(dim_tiempo.head())
print(f"Registros en DIM_TIEMPO: {len(dim_tiempo):,}")

,id_tiempo,fecha,anio,trimestre,mes,nombre_mes,dia,dia_semana_num,nombre_dia_semana,es_fin_de_semana,semana_anio
0,20160101,2016-01-01,2016,1,1,January,1,5,Friday,False,53
1,20160102,2016-01-02,2016,1,1,January,2,6,Saturday,True,53
2,20160103,2016-01-03,2016,1,1,January,3,7,Sunday,True,53
3,20160104,2016-01-04,2016,1,1,January,4,1,Monday,False,1
4,20160105,2016-01-05,2016,1,1,January,5,2,Tuesday,False,1


Registros en DIM_TIEMPO: 3,229


# Dimensión de Gravedad

In [14]:
dim_gravedad = (
    df[
        [
            "tipo_confirmado_raw",
            "gravedad_estandarizada"
        ]
    ]
    .drop_duplicates("gravedad_estandarizada")
    .sort_values("gravedad_estandarizada")
    .reset_index(drop=True)
)

dim_gravedad.insert(
    0,
    "id_gravedad",
    range(1, len(dim_gravedad) + 1)
)

dim_gravedad["nivel_severidad"] = (
    dim_gravedad["gravedad_estandarizada"]
    .map({
        "NEGATIVO": 0,
        "SOLO DANOS": 1,
        "CON HERIDOS": 2,
        "CON FALLECIDOS": 4,
        "SIN DATO": 0,
        "OTRO": 0
    })
    .fillna(0)
    .astype(int)
)

dim_gravedad["es_siniestro_positivo"] = ~(
    dim_gravedad["gravedad_estandarizada"]
    .isin(["NEGATIVO", "SIN DATO"])
)

display(dim_gravedad)

,id_gravedad,tipo_confirmado_raw,gravedad_estandarizada,nivel_severidad,es_siniestro_positivo
0,1,CON FALLECIDO,CON FALLECIDOS,4,True
1,2,CON LESIONADO,CON HERIDOS,2,True
2,3,NEGATIVO,NEGATIVO,0,False
3,4,CON MUERTOS,OTRO,0,True
4,5,SOLO DAÑOS,SOLO DANOS,1,True


In [15]:
## Se crea la relación con la tabla de hechos 

mapa_gravedad = dict(
    zip(
        dim_gravedad["gravedad_estandarizada"],
        dim_gravedad["id_gravedad"]
    )
)

df["id_gravedad"] = df["gravedad_estandarizada"].map(mapa_gravedad)

# Dimensión Tipo Accidentes

In [17]:
dim_tipo_accidente = (
    df[
        [
            "tipo_accidente_raw",
            "tipo_accidente_estandarizado"
        ]
    ]
    .drop_duplicates("tipo_accidente_estandarizado")
    .sort_values("tipo_accidente_estandarizado")
    .reset_index(drop=True)
)

dim_tipo_accidente.insert(
    0,
    "id_tipo_accidente",
    range(1, len(dim_tipo_accidente) + 1)
)

display(dim_tipo_accidente)

,id_tipo_accidente,tipo_accidente_raw,tipo_accidente_estandarizado
0,1,ATROPELLO,ATROPELLO
1,2,CA?DA OCUPANTE,CAIDA DE OCUPANTE
2,3,CHOQUE,CHOQUE
3,4,INCENDIO,INCENDIO
4,5,OTRO,OTRO
5,6,<NA>,SIN DATO
6,7,VOLCAMIENTO,VOLCAMIENTO


In [18]:
## Se crea la relación con la tabla de hechos 
mapa_accidente = dict(
    zip(
        dim_tipo_accidente["tipo_accidente_estandarizado"],
        dim_tipo_accidente["id_tipo_accidente"]
    )
)

df["id_tipo_accidente"] = (
    df["tipo_accidente_estandarizado"]
    .map(mapa_accidente)
)

# Dimensión Medio Reporte

In [19]:
dim_medio_reporte = (
    df[
        [
            "medio_reporte",
            "medio_reporte_estandarizado"
        ]
    ]
    .drop_duplicates("medio_reporte_estandarizado")
    .sort_values("medio_reporte_estandarizado")
    .reset_index(drop=True)
)

dim_medio_reporte.insert(
    0,
    "id_medio_reporte",
    range(1, len(dim_medio_reporte) + 1)
)

dim_medio_reporte["es_reporte_agente"] = (
    dim_medio_reporte["medio_reporte_estandarizado"]
    == "AGENTE DE TRANSITO"
)

display(dim_medio_reporte)

,id_medio_reporte,medio_reporte,medio_reporte_estandarizado,es_reporte_agente
0,1,AGENTE TRACR,AGENTE DE TRANSITO,True
1,2,AGENTE CAD 123,LINEA 123,False
2,3,TELEF?NICO 127,LINEA 127,False
3,4,MOVILIDAD CALI,OTRO-NARRATIVA,False
4,5,RADIO,RADIO,False
5,6,REDES SOCIALES,REDES SOCIALES,False
6,7,<NA>,SIN DATO,False


In [20]:
mapa_medio = dict(
    zip(
        dim_medio_reporte["medio_reporte_estandarizado"],
        dim_medio_reporte["id_medio_reporte"]
    )
)

df["id_medio_reporte"] = (
    df["medio_reporte_estandarizado"]
    .map(mapa_medio)
)

# Dimensión Ubicación

In [21]:
dim_ubicacion = (
    df[["direccion_reporte"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_ubicacion["direccion_reporte"] = (
    dim_ubicacion["direccion_reporte"]
    .fillna("SIN DATO")
)

dim_ubicacion["tipo_via_principal"] = np.select(
    [
        dim_ubicacion["direccion_reporte"].str.startswith("CARRERA"),
        dim_ubicacion["direccion_reporte"].str.startswith("CALLE"),
        dim_ubicacion["direccion_reporte"].str.startswith("DIAGONAL"),
        dim_ubicacion["direccion_reporte"].str.startswith("TRANSVERSAL"),
        dim_ubicacion["direccion_reporte"].str.startswith("AVENIDA"),
        dim_ubicacion["direccion_reporte"].eq("SIN DATO")
    ],
    [
        "CARRERA",
        "CALLE",
        "DIAGONAL",
        "TRANSVERSAL",
        "AVENIDA",
        "SIN DATO"
    ],
    default="OTRO"
)

dim_ubicacion["es_interseccion"] = (
    dim_ubicacion["direccion_reporte"]
    .str.contains(r"\bCON\b", regex=True, na=False)
)

dim_ubicacion.insert(
    0,
    "id_ubicacion",
    range(1, len(dim_ubicacion) + 1)
)

display(dim_ubicacion.head())
print(f"Registros en DIM_UBICACION: {len(dim_ubicacion):,}")

,id_ubicacion,direccion_reporte,tipo_via_principal,es_interseccion
0,1,CALLE 9C & CARRERA 29A,CALLE,False
1,2,CARRERA 94 CON CALLE 4 C,CARRERA,True
2,3,CALLE 16 & CARRERA 50,CALLE,False
3,4,CALLE 73 & CARRERA 26F,CALLE,False
4,5,CALLE 46 & CARRERA 43B,CALLE,False


Registros en DIM_UBICACION: 59,360


In [22]:
mapa_ubicacion = dict(
    zip(
        dim_ubicacion["direccion_reporte"],
        dim_ubicacion["id_ubicacion"]
    )
)

df["id_ubicacion"] = (
    df["direccion_reporte"]
    .fillna("SIN DATO")
    .map(mapa_ubicacion)
)

# Dimensión Tipo Vehículo

In [23]:
## Se separan los tipos
vehiculos = (
    df[
        [
            "id_siniestro",
            "tipos_vehiculos"
        ]
    ]
    .dropna(subset=["tipos_vehiculos"])
    .assign(
        tipo_vehiculo_raw=lambda tabla: tabla["tipos_vehiculos"]
        .str.split(",")
    )
    .explode("tipo_vehiculo_raw")
)

vehiculos["tipo_vehiculo_raw"] = (
    vehiculos["tipo_vehiculo_raw"]
    .str.strip()
    .str.upper()
)

vehiculos = vehiculos[
    [
        "id_siniestro",
        "tipo_vehiculo_raw"
    ]
].drop_duplicates()

In [24]:
dim_tipo_vehiculo = (
    vehiculos[["tipo_vehiculo_raw"]]
    .drop_duplicates()
    .sort_values("tipo_vehiculo_raw")
    .reset_index(drop=True)
)

dim_tipo_vehiculo.insert(
    0,
    "id_tipo_vehiculo",
    range(1, len(dim_tipo_vehiculo) + 1)
)

texto_vehiculo = dim_tipo_vehiculo["tipo_vehiculo_raw"]

dim_tipo_vehiculo["tipo_vehiculo_estandarizado"] = (
    texto_vehiculo
    .str.replace("BUSPADRON", "BUS PADRON", regex=False)
)

dim_tipo_vehiculo["categoria_vehiculo"] = np.select(
    [
        texto_vehiculo.str.contains("MOTO", na=False),
        texto_vehiculo.str.contains("BICI", na=False),
        texto_vehiculo.str.contains(
            "CAMION|BUS|TRACTOCAMION",
            na=False
        ),
        texto_vehiculo.str.contains(
            "MAQUINARIA|TRACTOR",
            na=False
        ),
        texto_vehiculo.str.contains(
            "AUTOMOVIL|TAXI|CAMPERO",
            na=False
        )
    ],
    [
        "MOTOCICLETA",
        "NO MOTORIZADO",
        "PESADO",
        "MAQUINARIA",
        "LIVIANO"
    ],
    default="OTRO"
)

display(dim_tipo_vehiculo.head())

,id_tipo_vehiculo,tipo_vehiculo_raw,tipo_vehiculo_estandarizado,categoria_vehiculo
0,1,"""AMBULANCIA CAMPERO""","""AMBULANCIA CAMPERO""",LIVIANO
1,2,"""AUTOMOVIL TURISMO""","""AUTOMOVIL TURISMO""",LIVIANO
2,3,"""AUTOMOVILTURISMO""","""AUTOMOVILTURISMO""",LIVIANO
3,4,"""BUS ALIMENTADOR""","""BUS ALIMENTADOR""",PESADO
4,5,"""BUS ARTICULADO""","""BUS ARTICULADO""",PESADO


In [25]:
mapa_vehiculo = dict(
    zip(
        dim_tipo_vehiculo["tipo_vehiculo_raw"],
        dim_tipo_vehiculo["id_tipo_vehiculo"]
    )
)

bridge_siniestro_vehiculo = vehiculos.copy()

bridge_siniestro_vehiculo["id_tipo_vehiculo"] = (
    bridge_siniestro_vehiculo["tipo_vehiculo_raw"]
    .map(mapa_vehiculo)
)

cantidad_tipos = (
    bridge_siniestro_vehiculo
    .groupby("id_siniestro")["id_tipo_vehiculo"]
    .transform("nunique")
)

bridge_siniestro_vehiculo["peso"] = 1 / cantidad_tipos

bridge_siniestro_vehiculo = bridge_siniestro_vehiculo[
    [
        "id_siniestro",
        "id_tipo_vehiculo",
        "peso"
    ]
]

display(bridge_siniestro_vehiculo.head())

,id_siniestro,id_tipo_vehiculo,peso
370987,AI013532,42,0.5
370987,AI013532,21,0.5
2,96161,42,1.0
370988,AI013533,21,0.5
370988,AI013533,42,0.5


# Puente Siniestro-Vehículo

In [26]:
mapa_vehiculo = dict(
    zip(
        dim_tipo_vehiculo["tipo_vehiculo_raw"],
        dim_tipo_vehiculo["id_tipo_vehiculo"]
    )
)

bridge_siniestro_vehiculo = vehiculos.copy()

bridge_siniestro_vehiculo["id_tipo_vehiculo"] = (
    bridge_siniestro_vehiculo["tipo_vehiculo_raw"]
    .map(mapa_vehiculo)
)

cantidad_tipos = (
    bridge_siniestro_vehiculo
    .groupby("id_siniestro")["id_tipo_vehiculo"]
    .transform("nunique")
)

bridge_siniestro_vehiculo["peso"] = 1 / cantidad_tipos

bridge_siniestro_vehiculo = bridge_siniestro_vehiculo[
    [
        "id_siniestro",
        "id_tipo_vehiculo",
        "peso"
    ]
]

display(bridge_siniestro_vehiculo.head())

,id_siniestro,id_tipo_vehiculo,peso
370987,AI013532,42,0.5
370987,AI013532,21,0.5
2,96161,42,1.0
370988,AI013533,21,0.5
370988,AI013533,42,0.5


In [27]:
print("DIM_TIEMPO:", len(dim_tiempo))
print("DIM_GRAVEDAD:", len(dim_gravedad))
print("DIM_TIPO_ACCIDENTE:", len(dim_tipo_accidente))
print("DIM_UBICACION:", len(dim_ubicacion))
print("DIM_MEDIO_REPORTE:", len(dim_medio_reporte))
print("DIM_TIPO_VEHICULO:", len(dim_tipo_vehiculo))
print("BRIDGE:", len(bridge_siniestro_vehiculo))

print("\nClaves faltantes:")
print("id_tiempo:", df["fecha"].isna().sum())
print("id_gravedad:", df["id_gravedad"].isna().sum())
print("id_tipo_accidente:", df["id_tipo_accidente"].isna().sum())
print("id_ubicacion:", df["id_ubicacion"].isna().sum())
print("id_medio_reporte:", df["id_medio_reporte"].isna().sum())

DIM_TIEMPO: 3229
DIM_GRAVEDAD: 5
DIM_TIPO_ACCIDENTE: 7
DIM_UBICACION: 59360
DIM_MEDIO_REPORTE: 7
DIM_TIPO_VEHICULO: 54
BRIDGE: 219711

Claves faltantes:
id_tiempo: 0
id_gravedad: 0
id_tipo_accidente: 0
id_ubicacion: 0
id_medio_reporte: 0


# Crear la tabla de hechos Fact_Siniestro

In [28]:
df["id_tiempo"] = (
    df["fecha"]
    .dt.strftime("%Y%m%d")
    .astype("Int64")
)

In [29]:
fact_siniestro = df[
    [
        "id_siniestro",
        "id_tiempo",
        "id_gravedad",
        "id_tipo_accidente",
        "id_ubicacion",
        "id_medio_reporte",
        "numero_vehiculos"
    ]
].copy()

fact_siniestro["cantidad_siniestros"] = 1

display(fact_siniestro.head())
print(f"Registros en FACT_SINIESTRO: {len(fact_siniestro):,}")

,id_siniestro,id_tiempo,id_gravedad,id_tipo_accidente,id_ubicacion,id_medio_reporte,numero_vehiculos,cantidad_siniestros
370987,AI013532,20241102,3,3,1,2,2,1
2,96161,20160101,2,1,2,3,2,1
370988,AI013533,20241102,2,3,3,3,2,1
370980,AI013525,20241102,3,3,4,3,2,1
370985,AI013530,20241102,2,3,5,3,2,1


Registros en FACT_SINIESTRO: 370,998


In [30]:
print("Filas del DataFrame original:", len(df))
print("Filas de FACT_SINIESTRO:", len(fact_siniestro))
print("Siniestros duplicados:", fact_siniestro["id_siniestro"].duplicated().sum())

print("\nClaves faltantes en la tabla de hechos:")
display(fact_siniestro.isna().sum())

Filas del DataFrame original: 370998
Filas de FACT_SINIESTRO: 370998
Siniestros duplicados: 0

Claves faltantes en la tabla de hechos:


id_siniestro                0
id_tiempo                   0
id_gravedad                 0
id_tipo_accidente           0
id_ubicacion                0
id_medio_reporte            0
numero_vehiculos       194421
cantidad_siniestros         0
dtype: int64

# Conexión PostgreSQL

In [25]:
%pip install pandas sqlalchemy psycopg2-binary python-dotenv

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
    --------------------------------------- 0.0/2.4 MB 1.3 MB/s eta 0:00:02
    --------------------------------------- 0.0/2.4 MB 653.6 kB/s eta 0:00:04
   - -------------------------------------- 0.1/2.4 MB 655.4 kB/s eta 0:00:04
   -- ------------------------------------- 0.2/2.4 MB 984.6 kB/s eta 0:00:03
   --- ------------------------------------ 0.2/2.4 MB 981.9 kB/s eta 0:00:03
   ----- ---------------------------------- 0.3/2.4 MB 1.2 MB/s eta 0:00:02
   ------- -------------------------------- 0.5/2.4 MB 1.5 MB/s eta 0:00:02
   ---------- ----------------------------- 0.7/2.4 MB 1.8 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.4 MB 2.4 MB/s eta 0:00:01
   -------------------- ------------------- 1.2/2.4 MB 2.7 MB/s eta 0:00:01
   ------------------------- -------------- 1.5/2.4 MB 3.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.9/2.4 MB 3.4 MB/s eta 0:00:01
   --------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import os
from pathlib import Path

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError(
        "No se encontró DATABASE_URL en el archivo .env."
    )

try:
    engine = create_engine(DATABASE_URL)

    with engine.connect() as conexion:
        conexion.execute(text("SELECT 1"))

    print("Conexión exitosa a PostgreSQL.")

except SQLAlchemyError as error:
    raise ConnectionError(
        "No fue posible conectarse a PostgreSQL. "
        "Verifica que el servidor esté activo, que la base de datos exista "
        "y que DATABASE_URL sea correcta."
    ) from error

Conexión exitosa a PostgreSQL.


# Carga de los DataFrames

In [33]:
tablas_carga = {
    "dim_tiempo": dim_tiempo,
    "dim_gravedad": dim_gravedad,
    "dim_tipo_accidente": dim_tipo_accidente,
    "dim_ubicacion": dim_ubicacion,
    "dim_medio_reporte": dim_medio_reporte,
    "dim_tipo_vehiculo": dim_tipo_vehiculo,
    "fact_siniestro": fact_siniestro,
    "bridge_siniestro_vehiculo": bridge_siniestro_vehiculo
}

try:
    for nombre_tabla, tabla in tablas_carga.items():
        tabla.to_sql(
            nombre_tabla,
            con=engine,
            if_exists="replace",
            index=False,
            method="multi",
            chunksize=5000
        )

        print(f"Tabla cargada correctamente: {nombre_tabla}")

except SQLAlchemyError as error:
    raise RuntimeError(
        f"Error durante la carga de datos: {error}"
    ) from error

Tabla cargada correctamente: dim_tiempo
Tabla cargada correctamente: dim_gravedad
Tabla cargada correctamente: dim_tipo_accidente
Tabla cargada correctamente: dim_ubicacion
Tabla cargada correctamente: dim_medio_reporte
Tabla cargada correctamente: dim_tipo_vehiculo
Tabla cargada correctamente: fact_siniestro
Tabla cargada correctamente: bridge_siniestro_vehiculo


In [34]:
## Verificar si cargaron los documentos

with engine.connect() as conexion:
    for nombre_tabla in tablas_carga:
        cantidad = conexion.execute(
            text(f'SELECT COUNT(*) FROM "{nombre_tabla}"')
        ).scalar_one()

        print(f"{nombre_tabla}: {cantidad:,} registros")

dim_tiempo: 3,229 registros
dim_gravedad: 5 registros
dim_tipo_accidente: 7 registros
dim_ubicacion: 59,360 registros
dim_medio_reporte: 7 registros
dim_tipo_vehiculo: 54 registros
fact_siniestro: 370,998 registros
bridge_siniestro_vehiculo: 219,711 registros


In [35]:
with engine.connect() as conexion:
    resultado = conexion.execute(
        text("SELECT current_database(), current_user")
    ).fetchone()

print(f"Base de datos utilizada: {resultado[0]}")
print(f"Usuario utilizado: {resultado[1]}")
print("Proceso ETL finalizado correctamente.")

Base de datos utilizada: siniestralidad_cali
Usuario utilizado: postgres
Proceso ETL finalizado correctamente.


# Consultas SQL Analiticas

¿Qué semanas del año concentran la mayor cantidad de siniestros?

In [39]:
q1 = '''
SELECT
    t.anio,
    t.semana_anio,
    SUM(f.cantidad_siniestros) AS cantidad_siniestros,
    RANK() OVER (
        ORDER BY SUM(f.cantidad_siniestros) DESC
    ) AS ranking
FROM fact_siniestro AS f
JOIN dim_tiempo AS t
    ON f.id_tiempo = t.id_tiempo
GROUP BY
    t.anio,
    t.semana_anio
ORDER BY
    ranking;
'''
r1 = pd.read_sql(q1, engine)
print("Consulta 1:")
print(r1)

Consulta 1:
     anio  semana_anio  cantidad_siniestros  ranking
0    2016           51               1575.0        1
1    2016           50               1565.0        2
2    2017            6               1512.0        3
3    2016           44               1465.0        4
4    2017            5               1423.0        5
..    ...          ...                  ...      ...
458  2024            1                181.0      458
459  2020           14                180.0      460
460  2020           13                132.0      461
461  2021           53                 96.0      462
462  2021           18                 54.0      463

[463 rows x 4 columns]


## Interpretacion 
Las primeras filas corresponden a las semanas con mayor cantidad de siniestros registrados. El RANK() permite identificar fácilmente las semanas que ocupan las primeras posiciones, teniendo en cuenta el año para no mezclar semanas de diferentes años.

¿Qué mes presenta la menor cantidad de siniestros registrados?

In [40]:
q2 = '''
-- Q2
SELECT
    t.mes,
    t.nombre_mes,
    SUM(f.cantidad_siniestros) AS cantidad_siniestros
FROM fact_siniestro AS f
JOIN dim_tiempo AS t
    ON f.id_tiempo = t.id_tiempo
GROUP BY
    t.mes,
    t.nombre_mes
ORDER BY
    cantidad_siniestros ASC;
'''
r2 = pd.read_sql(q2, engine)
print("Consulta 2:")
print(r2)

Consulta 2:
    mes nombre_mes  cantidad_siniestros
0     4      April              28366.0
1     6       June              28640.0
2     5        May              29143.0
3     1    January              29450.0
4     7       July              29456.0
5    11   November              30377.0
6     8     August              31508.0
7    10    October              32340.0
8     9  September              32470.0
9     2   February              32983.0
10   12   December              33044.0
11    3      March              33221.0


## interpretacion
El resultado permite identificar qué meses concentran menos siniestros durante el período analizado. La comparación se realiza acumulando los registros de todos los años disponibles para cada número de mes.

¿En qué semanas se registran más siniestros asociados a motocicletas que a automóviles?

In [47]:
q3 = '''
SELECT
    t.anio,
    t.semana_anio,
    COUNT(DISTINCT CASE
        WHEN v.categoria_vehiculo = 'MOTOCICLETA'
        THEN f.id_siniestro
    END) AS siniestros_motocicleta,
    COUNT(DISTINCT CASE
        WHEN v.tipo_vehiculo_estandarizado LIKE '%AUTOMOVIL%'
        THEN f.id_siniestro
    END) AS siniestros_automovil
FROM fact_siniestro AS f
JOIN dim_tiempo AS t
    ON f.id_tiempo = t.id_tiempo
JOIN bridge_siniestro_vehiculo AS b
    ON f.id_siniestro = b.id_siniestro
JOIN dim_tipo_vehiculo AS v
    ON b.id_tipo_vehiculo = v.id_tipo_vehiculo
GROUP BY
    t.anio,
    t.semana_anio
HAVING
    COUNT(DISTINCT CASE
        WHEN v.categoria_vehiculo = 'MOTOCICLETA'
        THEN f.id_siniestro
    END)
    >
    COUNT(DISTINCT CASE
        WHEN v.tipo_vehiculo_estandarizado LIKE '%AUTOMOVIL%'
        THEN f.id_siniestro
    END)
ORDER BY
    t.anio,
    t.semana_anio;

'''

r3 = pd.read_sql_query(text(q3), engine,params={"patron_auto": "%AUTOMOVIL%"})
print("Consulta 3:")
print(r3)

Consulta 3:
    anio  semana_anio  siniestros_motocicleta  siniestros_automovil
0   2016           53                      25                     6
1   2020           13                       2                     0
2   2020           15                      52                    49
3   2022           48                      92                    90
4   2023            4                      65                    54
..   ...          ...                     ...                   ...
83  2024           40                      88                    65
84  2024           41                      80                    63
85  2024           42                      99                    89
86  2024           43                     101                    75
87  2024           44                      84                    61

[88 rows x 4 columns]


## interpretacion
La tabla muestra únicamente las semanas en las que los siniestros asociados a motocicletas superan a los asociados a automóviles. Se utiliza COUNT(DISTINCT id_siniestro) porque un mismo siniestro puede involucrar varios vehículos y existe una tabla puente para representar esa relación.

¿Qué día de la semana concentra la mayor cantidad de siniestros?

In [42]:
q4='''
SELECT
    t.dia_semana_num,
    t.nombre_dia_semana,
    SUM(f.cantidad_siniestros) AS cantidad_siniestros
FROM fact_siniestro AS f
JOIN dim_tiempo AS t
    ON f.id_tiempo = t.id_tiempo
GROUP BY
    t.dia_semana_num,
    t.nombre_dia_semana
ORDER BY
    cantidad_siniestros DESC;
'''
r4 = pd.read_sql(q4, engine)
print("Consulta 4:")
print(r4) 

Consulta 4:
   dia_semana_num nombre_dia_semana  cantidad_siniestros
0               5            Friday              58673.0
1               2           Tuesday              56283.0
2               6          Saturday              55358.0
3               4          Thursday              55158.0
4               3         Wednesday              54906.0
5               1            Monday              51253.0
6               7            Sunday              39367.0


## interpretacion
El primer registro corresponde al día de la semana que concentra la mayor cantidad de siniestros. Esta consulta permite identificar patrones temporales relacionados con los días de la semana durante todo el período analizado.

¿Qué tipo de vehículo aparece con mayor frecuencia en los siniestros según la semana del año?

In [43]:
q5= '''
WITH frecuencia_vehiculos AS (
    SELECT
        t.anio,
        t.semana_anio,
        v.categoria_vehiculo,
        COUNT(DISTINCT f.id_siniestro) AS cantidad_siniestros
    FROM fact_siniestro AS f
    JOIN dim_tiempo AS t
        ON f.id_tiempo = t.id_tiempo
    JOIN bridge_siniestro_vehiculo AS b
        ON f.id_siniestro = b.id_siniestro
    JOIN dim_tipo_vehiculo AS v
        ON b.id_tipo_vehiculo = v.id_tipo_vehiculo
    GROUP BY
        t.anio,
        t.semana_anio,
        v.categoria_vehiculo
),
ranking_vehiculos AS (
    SELECT
        anio,
        semana_anio,
        categoria_vehiculo,
        cantidad_siniestros,
        ROW_NUMBER() OVER (
            PARTITION BY anio, semana_anio
            ORDER BY cantidad_siniestros DESC
        ) AS posicion
    FROM frecuencia_vehiculos
)
SELECT
    anio,
    semana_anio,
    categoria_vehiculo,
    cantidad_siniestros
FROM ranking_vehiculos
WHERE posicion = 1
ORDER BY
    anio,
    semana_anio;
'''
r5 = pd.read_sql(q5, engine)
print("Consulta 5:")
print(r5) 

Consulta 5:
     anio  semana_anio categoria_vehiculo  cantidad_siniestros
0    2016            1            LIVIANO                  150
1    2016            2            LIVIANO                  107
2    2016            3            LIVIANO                  169
3    2016            4            LIVIANO                  119
4    2016            5            LIVIANO                  147
..    ...          ...                ...                  ...
458  2024           40        MOTOCICLETA                   88
459  2024           41        MOTOCICLETA                   80
460  2024           42        MOTOCICLETA                   99
461  2024           43        MOTOCICLETA                  101
462  2024           44        MOTOCICLETA                   84

[463 rows x 4 columns]


## interpretacion
La consulta identifica, para cada semana y año, la categoría de vehículo que aparece en mayor cantidad de siniestros. ROW_NUMBER() permite conservar únicamente la categoría que ocupa la primera posición dentro de cada semana.

¿Cómo varía la cantidad de siniestros según el tipo de accidente, la gravedad y el día de la semana?

In [45]:
q6='''
SELECT
    t.nombre_dia_semana,
    a.tipo_accidente_estandarizado,
    g.gravedad_estandarizada,
    SUM(f.cantidad_siniestros) AS cantidad_siniestros
FROM fact_siniestro AS f
JOIN dim_tiempo AS t
    ON f.id_tiempo = t.id_tiempo
JOIN dim_tipo_accidente AS a
    ON f.id_tipo_accidente = a.id_tipo_accidente
JOIN dim_gravedad AS g
    ON f.id_gravedad = g.id_gravedad
GROUP BY
    t.dia_semana_num,
    t.nombre_dia_semana,
    a.tipo_accidente_estandarizado,
    g.gravedad_estandarizada
ORDER BY
    t.dia_semana_num,
    cantidad_siniestros DESC;
'''
r6 = pd.read_sql(q6, engine)
print("Consulta 6:")
print(r6) 

Consulta 6:
    nombre_dia_semana tipo_accidente_estandarizado gravedad_estandarizada  \
0              Monday                     SIN DATO               NEGATIVO   
1              Monday                     SIN DATO             SOLO DANOS   
2              Monday                     SIN DATO            CON HERIDOS   
3              Monday                       CHOQUE               NEGATIVO   
4              Monday                       CHOQUE            CON HERIDOS   
..                ...                          ...                    ...   
160            Sunday                    ATROPELLO             SOLO DANOS   
161            Sunday                  VOLCAMIENTO             SOLO DANOS   
162            Sunday                  VOLCAMIENTO         CON FALLECIDOS   
163            Sunday            CAIDA DE OCUPANTE             SOLO DANOS   
164            Sunday            CAIDA DE OCUPANTE               NEGATIVO   

     cantidad_siniestros  
0                30886.0  
1        